# 02 · DIFF vs BOUNDARY  —  is the leftover gated by a completion clock `c`, and is `boundary = DIFF==0`?

- **C1** surface vs semantic done-set — does the query track the literal PROGRESS code or the *semantic* done-set (⇒ it carries `c`, H3)?
- **C2** layer-emergence — at which GOAL layer band does the frontier first verbalize?
- **D1** reference readout of the DIFF==0 / all-done state — what does "nothing left" verbalize as?
- **D2** backup-goal-heads — do BOUNDARY heads also read out the frontier?
- **D4** done-state transplant — inject the all-done goal-write into a mid-task pivot: does it terminate? (`DIFF==0 ⇒ boundary`)
- **D3** competition vs upstream *(PRELIMINARY)* — boundary-done injection suppresses the frontier; does boosting the goal write rescue it?

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")            # so `_diff_primitives` is importable
import torch, numpy as np
import _diff_primitives as P
P.setup("Qwen/Qwen2.5-Coder-32B-Instruct")   # loads model + head masks (once)
from _diff_primitives import *                # helper functions into scope
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP

In [ ]:
# ── multipredicate example: 3 INDEPENDENT predicates, one assignment line each ──
# Each item can be implemented in any order (all orderings correct) — the pivot choice is a
# genuine "which sub-task next", not forced by a data dependency.
ITEMS = {
    "a": ("the list of words longer than three characters", "a = [w for w in words if len(w) > 3]"),
    "b": ("the count of palindromic words",                 "b = sum(w == w[::-1] for w in words)"),
    "c": ("the length of the longest word",                 "c = max(len(w) for w in words)"),
}
ORDER = ["a", "b", "c"]
TASK_SPEC = "\n".join(f"({i+1}) `{k}`: {ITEMS[k][0]}" for i, k in enumerate(ORDER))
QUESTION = ("Write a Python program that implements a function `func` which, given a list of "
            "strings `words`, computes:\n" + TASK_SPEC + "\n")

def build_prefill(done, extra=""):
    """Prefill implementing the assignments in `done` (kept in ORDER), pivot at the next indent."""
    lines = ["<code>", "def func(words: list[str]):"]
    for k in ORDER:
        if k in done:
            lines.append(f"    {ITEMS[k][1]}")
    return "\n".join(lines) + "\n    " + extra   # trailing indent = pivot sits here (about to write next var)

def build_prompt(done, extra=""):
    return get_prompt(QUESTION, build_prefill(done, extra))

def pivot_of(prompt):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] - 1

# task-item token spans are IDENTICAL across done-sets (task precedes the prefill) — compute once.
_p0 = build_prompt([])
ITEM_SPAN = {k: token_span(_p0, ITEMS[k][0]) for k in ORDER}
show_prompt("done={}      ", _p0, ITEM_SPAN["a"])
show_prompt("done={a}     ", build_prompt(["a"]), ITEM_SPAN["b"])
print("item spans:", ITEM_SPAN, " pivot(∅)=", pivot_token(_p0))

In [ ]:
# canonical mid-task instance: a done, frontier = b ; and the all-done (DIFF==0) instance.
p_mid  = build_prompt(["a"]);            piv_mid  = pivot_of(p_mid)
p_done = build_prompt(["a", "b", "c"], extra="return");  piv_done = pivot_of(p_done)
show_prompt("MID (frontier=b)", p_mid,  ITEM_SPAN["b"])
show_prompt("DONE (DIFF==0)  ", p_done, ITEM_SPAN["c"])
print("pivots: mid=", pivot_token(p_mid), " done=", pivot_token(p_done))

## C1 · Surface vs semantic done-set

Same surface code for item a, different done-*semantics*: (i) **live** `a = …` (bound), (ii) **dead**
`_ = …` (computed, not bound to the expected var), (iii) **commented** `# a = …` (surface-present, clearly
not done). Readout + next-var generation. **Tracks surface** ⇒ pure H1.1; **tracks semantics** (dead/comment
still treat a as undone) ⇒ query carries the completion clock `c` (H3).

In [ ]:
IMPL_A = ITEMS["a"][1]
variants = {
    "live  a = …  ": build_prompt(["a"]),
    "dead  _ = …  ": get_prompt(QUESTION, "<code>\ndef func(words: list[str]):\n    " + IMPL_A.replace("a =", "_ =", 1) + "\n    "),
    "comment # a=…": get_prompt(QUESTION, "<code>\ndef func(words: list[str]):\n    # " + IMPL_A + "\n    "),
    "none  (∅)    ": build_prompt([]),
}
for tag, p in variants.items():
    piv = pivot_of(p)
    g = greedy(prepare_input(prompts=p, tokenizer=mt), max_new=24)
    rich_print(f"[bold]{tag}[/bold]  next-gen => [italic]{escape(repr(g))}[/italic]")
    goalscope_readout("   readout ", p, piv, heads=GOAL_HEADS, maxnew=48)

## C2 · Layer-emergence of the frontier

Restrict the GOAL readout heads to increasing layer bands; see where the frontier (item b) first
verbalizes. Emerges only in the upper band ⇒ computed within the goal-head band; already present low ⇒
retained from upstream.

In [ ]:
for lo, hi in [(25, 35), (35, 45), (45, 56)]:
    hb = [(l, h) for (l, h) in GOAL_HEADS if lo <= l < hi]
    if hb:
        goalscope_readout(f"GOAL band {lo}-{hi} ({len(hb):2d}h)", p_mid, piv_mid, heads=hb, maxnew=48)

## D1 · Reference readout of the DIFF==0 (all-done) state

What does the goal-head readout say when nothing is left? This is the reference for D3/D4 —
"boundary" behaviour. **Predict:** "return / done / nothing left", not a task item.

In [ ]:
goalscope_readout("DONE (DIFF==0) ", p_done, piv_done, heads=GOAL_HEADS, maxnew=64)
rich_print(f"[grey70]clean DONE next-gen (expect return):[/grey70] "
           f"{escape(repr(greedy(prepare_input(prompts=p_done, tokenizer=mt), max_new=24)))}")

## D2 · Backup-goal-heads check

Read the frontier off the **BOUNDARY** heads instead of the GOAL heads (same mid-task pivot). If they
verbalize the frontier ≈ as well ⇒ redundant carriers (leans H2; predicts goal-head ablation won't kill
DIFF). If they don't ⇒ BOUNDARY_PATCH acts by same-layer competition, not by carrying DIFF.

In [ ]:
goalscope_readout("GOAL heads     ", p_mid, piv_mid, heads=GOAL_HEADS,     maxnew=48)
goalscope_readout("BOUNDARY heads ", p_mid, piv_mid, heads=BOUNDARY_HEADS, maxnew=48)
goalscope_readout("UNION heads    ", p_mid, piv_mid, heads=UNION_HEADS,    maxnew=48)

## D4 · Done-state transplant  (`DIFF==0 ⇒ boundary`, causal)

Cache the aggregated GOAL-head write at the all-done pivot (DIFF==0), additively inject it into the
mid-task generation (frontier=b). **Predict:** the model terminates / returns early instead of writing b.

In [ ]:
w_done = head_write(p_done, GOAL_HEADS, [piv_done])[0]        # (n_embd,)  the DIFF==0 write
dest_mid = dict(prepare_input(prompts=p_mid, tokenizer=mt))
rich_print(f"[grey70]clean mid  (expect writes b):[/grey70] "
           f"{escape(repr(greedy(prepare_input(prompts=p_mid, tokenizer=mt), max_new=32)))}")
for a in (0.1, 0.2, 0.3):
    g = head_additive_generate(mt, dest_mid, w_done, layers=list(range(35, 46)),
                               max_new_tokens=32, strategy="additive_norm", alpha=a)
    rich_print(f"[salmon1]+done-write α={a} (expect →return):[/salmon1] {escape(repr(g))}")

## D3 · Competition vs upstream  *(PRELIMINARY)*

Inject the **BOUNDARY**-heads' all-done write into the mid-task run (mimicking BOUNDARY_PATCH → drive
toward DIFF==0). Then test **rescue** by additionally boosting the mid-task GOAL write. Rescue ⇒ same-layer
competition/superposition at the readout; no rescue ⇒ boundary is genuinely upstream (H2).
*Preliminary: single additive channel, α hand-set — refine if the direction looks right.*

In [ ]:
w_bnd_done = head_write(p_done, BOUNDARY_HEADS, [piv_done])[0]   # boundary "done" write
w_goal_mid = head_write(p_mid,  GOAL_HEADS,     [piv_mid])[0]    # frontier(b) write, for the boost
for tag, vec, a, col in [
    ("+boundary-done (suppress?)          ", w_bnd_done,              0.2, "salmon1"),
    ("+boundary-done +goal-boost (rescue?)", w_bnd_done + w_goal_mid, 0.2, "gold3"),
]:
    g = head_additive_generate(mt, dest_mid, vec, layers=list(range(35, 46)),
                               max_new_tokens=32, strategy="additive_norm", alpha=a)
    rich_print(f"[{col}]{tag}:[/{col}] {escape(repr(g))}")